In [2]:
import pandas as pd
import numpy as np
from thefuzz import process, fuzz

In [3]:
df = pd.read_csv('clear_data/processed_data.csv')
df

,mini,base_mm2,cost,faction,price,price_old
0,Warboss,40mm,75.0,Orks,55.20,55.20
1,Warboss In Mega Armour,50mm,80.0,Orks,40.02,40.02
2,Warboss On Warbike,100 x 40mm,75.0,Orks,55.20,55.20
3,Weirdboy,40mm,65.0,Orks,30.82,30.82
4,Big Mek In Mega Armour,40mm,90.0,Orks,67.62,67.62
...,...,...,...,...,...,...
2371,Defiler,160mm,NaN,Death_Guard,133.40,133.40
2372,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus,59.80,59.80
2373,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus,59.80,59.80
2374,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus,59.80,59.80


In [4]:
mat = pd.read_csv('clear_data/df_materials.csv')
mat

,mini,faction,material,edition,year
0,Baharroth Wings,Craftworlds,Metal,Warhammer 40K: Rogue Trader,1990.0
1,Warp Spiders,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
2,Avatar of Khaine,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
3,Asurmen,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
4,Karandras,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
...,...,...,...,...,...
977,Ahriman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2016.0
978,Tzaangor Enlightened,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0
979,Tzaangor Skyfires,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0
980,Tzaangor Shaman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0


In [5]:
# Verificar coincidencias en 'mini' entre mat y df
common_minis = set(mat['mini']).intersection(set(df['mini']))
print(f"Número de coincidencias en 'mini': {len(common_minis)}")
print("Ejemplo de coincidencias:", list(common_minis)[:10])

Número de coincidencias en 'mini': 448
Ejemplo de coincidencias: ['Shroud Runners', 'Tactical Drones', 'Purgation Squad', 'Psychophage', 'Onager Dunecrawler', 'Forgefiend', 'Hospitaller', 'Yvraine', 'Battlewagon', 'Lokhust Heavy Destroyers']


In [6]:
def find_fuzzy_matches_fixed(source, target, threshold=85):
    matches = []
    for mini in source:
        result = process.extractOne(mini, target)
        if result:
            match, score = result[0], result[1]
            if score >= threshold:
                matches.append((mini, match, score))
    return matches

fuzzy_matches_fixed = find_fuzzy_matches_fixed(mat['mini'], df['mini'])
print(f"Número de coincidencias difusas corregidas: {len(fuzzy_matches_fixed)}")
print("Ejemplo de coincidencias difusas corregidas:")
for match in fuzzy_matches_fixed[:10]:
    print(match)

Número de coincidencias difusas corregidas: 946
Ejemplo de coincidencias difusas corregidas:
('Baharroth Wings', 'Baharroth', 90)
('Warp Spiders', 'WARP SPIDER', 96)
('Avatar of Khaine', 'Avatar of Khaine', 100)
('Asurmen', 'Asurmen', 100)
('Karandras', 'Karandras', 100)
('Fuegan', 'Fuegan', 100)
('Baharroth', 'Baharroth', 100)
('Maugan Ra', 'Maugan Ra', 100)
('Warp Spider Exarch', 'WARP SPIDER EXARCH', 100)
('Warlock with Witchblade', 'Warlock', 90)


In [7]:
# Crear un dataset combinado con todas las columnas de ambos DataFrames
combined_matches = []
for mini, match, score in fuzzy_matches_fixed:
    mat_row = mat.loc[mat['mini'] == mini]
    df_row = df.loc[df['mini'] == match]

    if not mat_row.empty and not df_row.empty:
        combined_row = {
            **mat_row.iloc[0].to_dict(),  # Todas las columnas de mat
            **df_row.iloc[0].to_dict(),  # Todas las columnas de df
            'match_score': score         # Agregar la puntuación de coincidencia
        }
        combined_matches.append(combined_row)

beta = pd.DataFrame(combined_matches)
print("Dataset combinado creado con todas las columnas de ambos DataFrames:")
beta

Dataset combinado creado con todas las columnas de ambos DataFrames:


,mini,faction,material,edition,year,base_mm2,cost,price,price_old,match_score
0,Baharroth,Craftworlds,Metal,Warhammer 40K: Rogue Trader,1990.0,40mm,115.0,44.16,44.16,90
1,WARP SPIDER,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,28.5mm,95.0,57.50,57.50,96
2,Avatar of Khaine,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,80mm,280.0,104.88,104.88,100
3,Asurmen,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,40mm,125.0,44.16,44.16,100
4,Karandras,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,25mm,90.0,52.90,52.90,100
...,...,...,...,...,...,...,...,...,...,...
941,Ahriman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2016.0,40mm,100.0,44.16,44.16,100
942,Tzaangor Enlightened,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,55.0,55.20,55.20,100
943,Tzaangor Enlightened with Fatecaster Greatbows,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,60.0,55.20,55.20,86
944,Tzaangor Shaman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,60.0,35.88,35.88,100


In [8]:
beta.to_csv('clear_data/df_W40K.csv', index=False)
beta

,mini,faction,material,edition,year,base_mm2,cost,price,price_old,match_score
0,Baharroth,Craftworlds,Metal,Warhammer 40K: Rogue Trader,1990.0,40mm,115.0,44.16,44.16,90
1,WARP SPIDER,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,28.5mm,95.0,57.50,57.50,96
2,Avatar of Khaine,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,80mm,280.0,104.88,104.88,100
3,Asurmen,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,40mm,125.0,44.16,44.16,100
4,Karandras,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0,25mm,90.0,52.90,52.90,100
...,...,...,...,...,...,...,...,...,...,...
941,Ahriman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2016.0,40mm,100.0,44.16,44.16,100
942,Tzaangor Enlightened,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,55.0,55.20,55.20,100
943,Tzaangor Enlightened with Fatecaster Greatbows,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,60.0,55.20,55.20,86
944,Tzaangor Shaman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0,40mm,60.0,35.88,35.88,100
